# Plackett-Luce Scoring

This notebook converts complete four-model expert rankings into
Plackett-Luce log-strengths, Elo-like scores, confidence intervals,
and pairwise win probabilities. Its public default execution is
deterministic and does not require the private evaluation TSV.

## Scope and Method

The calculation uses only `Gemini`, `Grok`, `OpenAI`, and `Phytomni`.
Other columns, including `Claude`, `Gene`, `Expert`, or `Species`, do
not enter the fit. Each accepted row supplies a complete best-to-worst
ordering. Gemini is fixed as the reference log-strength, and the other
parameters are estimated by maximizing the four-way Plackett-Luce
likelihood.

## Reproducibility Configuration

Set `DEEPGENOME_SCORE_TSV` to the private ranking TSV when it is
available. Set `DEEPGENOME_SAVE_RESULTS=1` only when the two result CSV
files should be written. With both variables unset, execution remains
offline and writes nothing.

In [ ]:
import os
from pathlib import Path


def optional_path(name: str) -> Path | None:
    value = os.getenv(name)
    return Path(value).expanduser().resolve() if value else None


SCORE_TSV = optional_path("DEEPGENOME_SCORE_TSV")
SAVE_RESULTS = os.getenv("DEEPGENOME_SAVE_RESULTS", "0") == "1"

## Load and Validate Rankings

Rank labels are accepted when they begin with `R` and the remaining
suffix parses as an integer. Suffixes are intentionally not restricted
to 1 through 4, and duplicate ranks retain the source-column order.
Rows missing a parseable rank for any target model are skipped and
counted.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize


MODEL_COLUMNS = ("Gemini", "Grok", "OpenAI", "Phytomni")
REFERENCE_MODEL = "Gemini"
OPTIMIZER_OPTIONS = {"ftol": 1e-10, "gtol": 1e-8, "maxiter": 1000}
HESSIAN_EPSILON = 1e-5
ELO_SCALE = 400.0 / np.log(10.0)
ELO_CENTER = 1500.0
CI_Z = 1.96


def parse_rankings(
    frame: pd.DataFrame,
    model_columns: tuple[str, ...] = MODEL_COLUMNS,
) -> tuple[list[list[str]], int]:
    missing = [column for column in model_columns if column not in frame.columns]
    if missing:
        raise ValueError(f"Missing required model columns: {', '.join(missing)}")
    rankings: list[list[str]] = []
    skipped = 0
    for _, row in frame.iterrows():
        rank_map: dict[str, int] = {}
        for model in model_columns:
            value = row[model]
            if isinstance(value, str) and value.startswith("R"):
                try:
                    rank_map[model] = int(value[1:])
                except ValueError:
                    continue
        if len(rank_map) != len(model_columns):
            skipped += 1
            continue
        ordered = sorted(rank_map.items(), key=lambda item: item[1])
        rankings.append([model for model, _ in ordered])
    return rankings, skipped


def pl_loglik_and_grad(
    xi: dict[str, float],
    rankings: list[list[str]],
    models: list[str],
    reference_model: str,
) -> tuple[float, np.ndarray]:
    free_models = [model for model in models if model != reference_model]
    index = {model: position for position, model in enumerate(free_models)}
    theta = {model: np.exp(xi[model]) for model in models}

    log_likelihood = 0.0
    gradient = np.zeros(len(free_models), dtype=float)

    for ranking in rankings:
        remaining = ranking[:]
        denominator = sum(theta[model] for model in remaining)
        log_likelihood += (
            np.log(theta[ranking[0]]) - np.log(denominator)
        )
        for model in remaining:
            if model != reference_model:
                if model == ranking[0]:
                    gradient[index[model]] += 1.0
                gradient[index[model]] -= theta[model] / denominator

        remaining = remaining[1:]
        denominator = sum(theta[model] for model in remaining)
        log_likelihood += (
            np.log(theta[ranking[1]]) - np.log(denominator)
        )
        for model in remaining:
            if model != reference_model:
                if model == ranking[1]:
                    gradient[index[model]] += 1.0
                gradient[index[model]] -= theta[model] / denominator

        remaining = remaining[1:]
        denominator = sum(theta[model] for model in remaining)
        log_likelihood += (
            np.log(theta[ranking[2]]) - np.log(denominator)
        )
        for model in remaining:
            if model != reference_model:
                if model == ranking[2]:
                    gradient[index[model]] += 1.0
                gradient[index[model]] -= theta[model] / denominator

    return log_likelihood, gradient


def pack_xi_vector(
    xi: dict[str, float],
    models: list[str],
    reference_model: str,
) -> np.ndarray:
    free_models = [model for model in models if model != reference_model]
    return np.array([xi[model] for model in free_models], dtype=float)


def unpack_xi_vector(
    vector: np.ndarray,
    models: list[str],
    reference_model: str,
) -> dict[str, float]:
    free_models = [model for model in models if model != reference_model]
    return {
        model: (
            0.0
            if model == reference_model
            else float(vector[free_models.index(model)])
        )
        for model in models
    }


def negative_pl_objective(
    vector: np.ndarray,
    rankings: list[list[str]],
    models: list[str],
    reference_model: str,
) -> tuple[float, np.ndarray]:
    xi = unpack_xi_vector(vector, models, reference_model)
    log_likelihood, gradient = pl_loglik_and_grad(
        xi,
        rankings,
        models,
        reference_model,
    )
    return -log_likelihood, -gradient


def central_difference_hessian(
    function,
    vector: np.ndarray,
    epsilon: float = HESSIAN_EPSILON,
) -> np.ndarray:
    size = len(vector)
    hessian = np.zeros((size, size))
    for index in range(size):
        plus = vector.copy()
        plus[index] += epsilon
        _, gradient_plus = function(plus)
        minus = vector.copy()
        minus[index] -= epsilon
        _, gradient_minus = function(minus)
        hessian[:, index] = (
            gradient_plus - gradient_minus
        ) / (2.0 * epsilon)
    return hessian


def covariance_from_hessian(
    hessian: np.ndarray,
    models: list[str],
    reference_model: str,
) -> np.ndarray:
    try:
        free_covariance = np.linalg.inv(hessian)
    except np.linalg.LinAlgError:
        free_covariance = np.linalg.pinv(hessian)
    covariance = np.zeros((len(models), len(models)))
    free_models = [model for model in models if model != reference_model]
    full_index = {model: index for index, model in enumerate(models)}
    for row, row_model in enumerate(free_models):
        for column, column_model in enumerate(free_models):
            covariance[
                full_index[row_model], full_index[column_model]
            ] = free_covariance[row, column]
    return covariance


def elo_from_xi(
    xi: dict[str, float],
    models: list[str],
) -> np.ndarray:
    xi_vector = np.array([xi[model] for model in models])
    elo_raw = ELO_SCALE * xi_vector
    offset = ELO_CENTER - np.mean(elo_raw)
    return elo_raw + offset


def elo_confidence_intervals(
    elo: np.ndarray,
    covariance: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    elo_standard_error = ELO_SCALE * np.sqrt(np.diag(covariance))
    elo_lower = elo - CI_Z * elo_standard_error
    elo_upper = elo + CI_Z * elo_standard_error
    return elo_standard_error, elo_lower, elo_upper


def pairwise_win_probabilities(
    xi: dict[str, float],
    models: list[str],
) -> np.ndarray:
    probabilities = np.zeros((len(models), len(models)))
    for row, row_model in enumerate(models):
        for column, column_model in enumerate(models):
            if row == column:
                probabilities[row, column] = np.nan
            else:
                probabilities[row, column] = 1.0 / (
                    1.0
                    + np.exp(xi[column_model] - xi[row_model])
                )
    return probabilities


def fit_plackett_luce(
    rankings: list[list[str]],
    models: list[str],
) -> dict[str, object]:
    models = sorted(models)
    reference_model = REFERENCE_MODEL
    if reference_model not in models:
        raise ValueError(
            f"Reference model {reference_model!r} is absent from the model list."
        )
    if not rankings:
        raise ValueError("At least one complete ranking is required.")

    initial_xi = {model: 0.0 for model in models}
    initial_vector = pack_xi_vector(
        initial_xi,
        models,
        reference_model,
    )

    def objective(vector: np.ndarray) -> tuple[float, np.ndarray]:
        return negative_pl_objective(
            vector,
            rankings,
            models,
            reference_model,
        )

    result = minimize(
        lambda vector: objective(vector)[0],
        initial_vector,
        jac=lambda vector: objective(vector)[1],
        method="L-BFGS-B",
        options=OPTIMIZER_OPTIONS,
    )
    xi_hat = unpack_xi_vector(result.x, models, reference_model)
    hessian = central_difference_hessian(
        objective,
        result.x,
        HESSIAN_EPSILON,
    )
    covariance = covariance_from_hessian(
        hessian,
        models,
        reference_model,
    )
    elo = elo_from_xi(xi_hat, models)
    elo_standard_error, elo_lower, elo_upper = (
        elo_confidence_intervals(elo, covariance)
    )
    pairwise_probabilities = pairwise_win_probabilities(xi_hat, models)

    return {
        "models": models,
        "reference_model": reference_model,
        "optimizer_result": result,
        "negative_log_likelihood": float(result.fun),
        "xi": xi_hat,
        "covariance": covariance,
        "elo": elo,
        "elo_standard_error": elo_standard_error,
        "elo_lower": elo_lower,
        "elo_upper": elo_upper,
        "pairwise_probabilities": pairwise_probabilities,
    }


def elo_outputs(
    fit: dict[str, object],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    models = fit["models"]
    score_table = pd.DataFrame(
        {
            "Model": models,
            "Elo": fit["elo"],
            "Elo_L": fit["elo_lower"],
            "Elo_U": fit["elo_upper"],
        }
    ).sort_values("Elo", ascending=False, ignore_index=True)
    probability_table = pd.DataFrame(
        fit["pairwise_probabilities"],
        index=models,
        columns=models,
    )
    return score_table, probability_table

In [ ]:
score_frame = None
benchmark_rankings: list[list[str]] = []
skipped_ranking_rows = 0

if SCORE_TSV is not None:
    if not SCORE_TSV.is_file():
        raise FileNotFoundError(f"Ranking TSV not found: {SCORE_TSV}")
    score_frame = pd.read_csv(SCORE_TSV, sep="	")
    benchmark_rankings, skipped_ranking_rows = parse_rankings(score_frame)
    print(f"Total ranking rows: {len(score_frame)}")
    print(f"Used ranking rows: {len(benchmark_rankings)}")
    print(f"Skipped ranking rows: {skipped_ranking_rows}")
    if not benchmark_rankings:
        raise ValueError("The ranking TSV contains no complete target-model rows.")

## Fit the Plackett-Luce Model

The free log-strengths start at zero and are optimized with L-BFGS-B.
A central-difference Hessian of the negative-objective gradient is
inverted for the historical covariance estimate, with a pseudo-inverse
fallback if direct inversion fails.

In [ ]:
benchmark_fit = None
if benchmark_rankings:
    benchmark_fit = fit_plackett_luce(
        benchmark_rankings,
        list(MODEL_COLUMNS),
    )
    optimizer_result = benchmark_fit["optimizer_result"]
    if not optimizer_result.success:
        print(f"WARNING: Optimization did not fully converge: {optimizer_result.message}")
    print(
        "Negative log-likelihood: "
        f"{benchmark_fit['negative_log_likelihood']:.12f}"
    )

## Elo-Like Scores and Uncertainty

Log-strengths are mapped with `400 / ln(10)` and centered to a mean
score of 1500. For numerical equivalence, uncertainty uses the historical
reference-parameter approximation: Gemini retains zero variance, and the
covariance is not transformed after Elo centering. The resulting interval
is therefore a compatibility calculation rather than a fully propagated
uncertainty interval.

In [ ]:
from IPython.display import display


score_table = None
probability_table = None
if benchmark_fit is not None:
    score_table, probability_table = elo_outputs(benchmark_fit)
    display(score_table)

## Pairwise Win Probabilities

Entry `(i, j)` is the logistic probability that row model `i` is
preferred to column model `j`. The matrix remains in alphabetical model
order, and its diagonal is undefined.

In [ ]:
if probability_table is not None:
    display(probability_table)

## Reproducibility Checks

The deterministic fixture below includes all 24 four-model permutations
once and six additional `Phytomni > OpenAI > Grok > Gemini` rankings. It
checks the likelihood, optimum, scores, uncertainty, and representative
pairwise probabilities before any missing-input status is reported.

In [ ]:
from itertools import permutations


fixture_models = list(MODEL_COLUMNS)
fixture_rankings = [
    list(order) for order in permutations(fixture_models)
]
fixture_rankings.extend(
    [["Phytomni", "OpenAI", "Grok", "Gemini"]] * 6
)
fixture_xi = {model: 0.0 for model in fixture_models}
fixture_log_likelihood, fixture_gradient = pl_loglik_and_grad(
    fixture_xi,
    fixture_rankings,
    fixture_models,
    REFERENCE_MODEL,
)
assert np.isclose(
    fixture_log_likelihood,
    -95.34161491043835,
    atol=1e-12,
)
np.testing.assert_allclose(
    fixture_gradient,
    [-0.5, 2.5, 4.5],
    atol=1e-12,
)

fixture_fit = fit_plackett_luce(fixture_rankings, fixture_models)
assert np.isclose(
    fixture_fit["negative_log_likelihood"],
    93.43927901604626,
    atol=1e-6,
)
np.testing.assert_allclose(
    [fixture_fit["xi"][model] for model in fixture_models],
    [0.0, 0.3126965119, 0.4923752789, 0.6228591999],
    atol=1e-6,
)
np.testing.assert_allclose(
    fixture_fit["elo"],
    [1437.9857450, 1492.3066929, 1523.5200917, 1546.1874704],
    atol=1e-6,
)
np.testing.assert_allclose(
    fixture_fit["elo_standard_error"],
    [0.0, 56.4574659, 58.3740115, 59.6279150],
    atol=1e-5,
)
fixture_probabilities = fixture_fit["pairwise_probabilities"]
assert np.isclose(fixture_probabilities[3, 0], 0.6508685493, atol=1e-6)
assert np.isclose(fixture_probabilities[3, 2], 0.5325747750, atol=1e-6)
assert np.isclose(fixture_probabilities[2, 1], 0.5447992300, atol=1e-6)
print("Passed: 30-ranking numerical equivalence fixture.")

if SCORE_TSV is None:
    print(
        "SKIPPED: DEEPGENOME_SCORE_TSV is unset; "
        "the private benchmark was not fitted."
    )

## Results and Optional Export

Current benchmark tables appear only when a valid ranking TSV was
configured. Exports retain the historical filenames and are written only
when `DEEPGENOME_SAVE_RESULTS=1`; the default execution emits no CSVs.

In [ ]:
if benchmark_fit is None:
    print("No private benchmark results are available for export.")
elif SAVE_RESULTS:
    score_table.to_csv("pl_elo_results.csv", index=False)
    probability_table.to_csv("pl_pairwise_probs.csv")
    print("Saved pl_elo_results.csv and pl_pairwise_probs.csv.")
else:
    print("Result export is disabled; set DEEPGENOME_SAVE_RESULTS=1 to enable it.")